# 套件

In [1]:
# prompt: 讀取雲端硬碟
import os
from google.colab import drive
drive.mount('/content/drive')


import requests
import pandas as pd
import random
import time

Mounted at /content/drive


In [2]:
stock_ids = ['00982A','00981A', '00878', '00919', '00918', '00940', '00929', '00713', '00939', '00881', '00900', '0050', '0056', '00934', '00932', '00923', '00915', '00936', '00733', '0057']

In [3]:
# prompt: 00878, 00919, 00940, 00929, 00713, 00939, 00881, 00900, 0050, 00918進行這個網頁的爬蟲
# url = "https://www.cmoney.tw/api/cm/MobileService/ashx/GetDtnoData.ashx?action=getdtnodata&DtNo=59449513&ParamStr=AssignID%3D{}%3BMTPeriod%3D0%3BDTMode%3D0%3BDTRange%3D1%3BDTOrder%3D1%3BMajorTable%3DM722%3B&FilterNo=0"
# stock_ids = ['00878']
# api_url = url.format(stock_id)
# 在每次for迴圈爬蟲時，設定一個隨機在5~9秒之間的暫停器，避免網頁過載
# 並打包成一個函式


def fetch_stock_data(stock_id):
  url = "https://www.cmoney.tw/api/cm/MobileService/ashx/GetDtnoData.ashx?action=getdtnodata&DtNo=59449513&ParamStr=AssignID%3D{}%3BMTPeriod%3D0%3BDTMode%3D0%3BDTRange%3D1%3BDTOrder%3D1%3BMajorTable%3DM722%3B&FilterNo=0"
  api_url = url.format(stock_id)
  res = requests.get(api_url)
  res_json = res.json()
  data_title = [res_json['Title'][0], res_json['Title'][1], res_json['Title'][3], res_json['Title'][4]]
  data =  [[d[0], d[1], d[3], d[4]] for d in res_json['Data']]
  df = pd.DataFrame(data, columns=data_title)
  df.columns = ['date', 'stock_id', 'weight(%)', 'holdings']
  df['holdings'] = df['holdings'].str.replace(',', '').astype('int64') / 1000
  df['etf'] = stock_id

  d = df['date'][0]
  print(stock_id, d)
  df.to_csv(f"//content//drive//MyDrive//etf_daily_holdings//{d}_{stock_id}.csv", encoding = 'utf-8_sig', index = False)
  return df


dfs = []
counter = 0
for stock_id in stock_ids:
  dfs.append(fetch_stock_data(stock_id))
  # Random sleep between 5 and 9 seconds
  sleep_time = random.randint( 9, 13)

  counter += 1
  #if counter == 2: break
  #break
  if counter < len(stock_ids):
    time.sleep(sleep_time)
  else:
    break

# Combine all dataframes into a single dataframe
df = pd.concat(dfs)
df

00982A 20260206
00981A 20260206
00878 20260206
00919 20260206
00918 20260206
00940 20260206
00929 20260206
00713 20260206
00939 20260206
00881 20260206
00900 20260206
0050 20260206
0056 20260206
00934 20260206
00932 20260206
00923 20260206
00915 20260206
00936 20260206
00733 20260206
0057 20260206


,date,stock_id,weight(%),holdings,etf
0,20260206,2330,8.84,1445.000,00982A
1,20260206,8299,5.56,854.000,00982A
2,20260206,6669,5.29,422.000,00982A
3,20260206,5536,5.27,2470.000,00982A
4,20260206,6139,3.92,2157.000,00982A
...,...,...,...,...,...
84,20260206,2633,0.07,8.703,0057
85,20260206,2105,0.07,7.878,0057
86,20260206,9910,0.06,2.183,0057
87,20260206,C_NTD,,995.633,0057


In [4]:

date = pd.to_datetime('today', format = '%Y%m%d').date()
path = "//content//drive//MyDrive//etf_daily_holdings//{}_{}.csv"

for stock_id in stock_ids:
  file_path = path.format(date, stock_id)
  print(file_path)

//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00982A.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00981A.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00878.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00919.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00918.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00940.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00929.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00713.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00939.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00881.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00900.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_0050.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_0056.csv
//content//drive//MyDrive//etf_daily_holdings//2026-02-07_00934.csv
//content//drive//MyDrive//etf_daily_holdings//2

## 設定持股變化參數

In [5]:
import pandas as pd
import os
date = pd.to_datetime('today', format = '%Y%m%d').date()
#date = pd.to_datetime('20240710', format = '%Y%m%d').date()
path = "//content//drive//MyDrive//etf_daily_holdings//{}_{}.csv"
offset_ = 1

stock_ids = ['00878', '00919', '00918', '00940', '00929', '00713', '00939', '00881', '00900', '0050', '0056', '00934', '00932', '00923', '00915', '00936', '00733']
df_delta = pd.DataFrame()
for stock_id in stock_ids:
  date = date
  print(date)
  file_path = path.format(date, stock_id)
  file_path = file_path.replace('-', '')

  while not os.path.isfile(file_path):
    print( file_path)
    date -= pd.DateOffset(1)
    file_path = path.format(date.date(), stock_id)
    file_path = file_path.replace('-', '')

    #break

  print(file_path)
  today_df = pd.read_csv(file_path)

  yesterday =  date - (pd.DateOffset(1))
  print(yesterday)
  yesterday_file_path = path.format(yesterday.date(), stock_id)
  yesterday_file_path = yesterday_file_path.replace('-', '')
  print(yesterday.date(), os.path.isfile(yesterday_file_path))
  while not os.path.isfile(yesterday_file_path):
    #
    yesterday -= pd.DateOffset(1)
    yesterday_file_path = path.format(yesterday.date(), stock_id)
    yesterday_file_path = yesterday_file_path.replace('-', '')
    #break
  print(yesterday_file_path)

  #print(date, yesterday)
  #print(yesterday_file_path)

  yesterday_df = pd.read_csv(yesterday_file_path)
  today_df = pd.merge(today_df, yesterday_df, on = ['stock_id', 'etf'], how = 'outer', suffixes = ('_today', '_yesterday'))
  today_df['holdings_today'] = today_df['holdings_today'].fillna(0)
  today_df['holdings_yesterday'] = today_df['holdings_yesterday'].fillna(0)
  today_df['holdings_diff'] = today_df['holdings_today'] -  today_df['holdings_yesterday']
  # save_path = "//content//drive//MyDrive//etf_daily_holdings//{}_{}.csv"
  # today_df.to_csv(file_path, encoding = 'utf-8_sig', index = False)
  df_delta = pd.concat([df_delta, today_df])
  print(file_path)
df_delta

2026-02-07
//content//drive//MyDrive//etf_daily_holdings//20260207_00878.csv
//content//drive//MyDrive//etf_daily_holdings//20260206_00878.csv
2026-02-05 00:00:00
2026-02-05 False
//content//drive//MyDrive//etf_daily_holdings//20260202_00878.csv
//content//drive//MyDrive//etf_daily_holdings//20260206_00878.csv
2026-02-06 00:00:00
//content//drive//MyDrive//etf_daily_holdings//20260206 00:00:00_00919.csv
//content//drive//MyDrive//etf_daily_holdings//20260205_00919.csv
//content//drive//MyDrive//etf_daily_holdings//20260204_00919.csv
//content//drive//MyDrive//etf_daily_holdings//20260203_00919.csv
//content//drive//MyDrive//etf_daily_holdings//20260202_00919.csv
2026-02-01 00:00:00
2026-02-01 False
//content//drive//MyDrive//etf_daily_holdings//20260130_00919.csv
//content//drive//MyDrive//etf_daily_holdings//20260202_00919.csv
2026-02-02 00:00:00
//content//drive//MyDrive//etf_daily_holdings//20260202 00:00:00_00918.csv
//content//drive//MyDrive//etf_daily_holdings//20260201_00918.csv

,date_today,stock_id,weight(%)_today,holdings_today,etf,date_yesterday,weight(%)_yesterday,holdings_yesterday,holdings_diff
0,20260206.0,1216,2.30,141500.000,878,20260202.0,2.24,141500.000,0.000
1,20260206.0,1402,1.56,255109.000,878,20260202.0,1.56,255109.000,0.000
2,20260206.0,202602TX,4.04,2.900,878,20260202.0,4.05,2.900,0.000
3,20260206.0,2105,0.92,137539.000,878,20260202.0,0.92,137539.000,0.000
4,20260206.0,2301,2.64,73233.000,878,20260202.0,2.60,73233.000,0.000
...,...,...,...,...,...,...,...,...,...
49,20251219.0,8404,0.20,536.000,733,20251217.0,0.20,537.000,-1.000
50,20251219.0,9910,3.72,2004.000,733,20251217.0,3.76,2008.000,-4.000
51,20251219.0,C_NTD,NaN,55894.453,733,20251217.0,NaN,51194.960,4699.493
52,20251219.0,M_NTD,NaN,70536.173,733,20251217.0,NaN,74580.887,-4044.714


# 持股變化

# 計算個股的 ETF 持股變化量

In [6]:
df_delta[(df_delta['etf'] == '00981A') | (df_delta['etf'] == '00982A')].groupby('stock_id')['holdings_today'].sum().sort_values(ascending = False)

,holdings_today
stock_id,


In [7]:
df_all = df_delta[(df_delta['stock_id'].str.len() == 4) ].groupby('stock_id')['holdings_today'].sum().sort_values(ascending = False)
df_all.to_csv("/content/drive/MyDrive/holdings.csv", encoding = 'utf-8_sig')


In [8]:
df_940 = df_delta[(df_delta['etf'] == 940) ].groupby('stock_id')['holdings_today'].sum().sort_values(ascending = False)#.to_csv("/content/drive/MyDrive/holdings.csv", encoding = 'utf-8_sig')
df_940_1 = df_delta[(df_delta['etf'] == 940) ].groupby('stock_id')['holdings_yesterday'].sum().sort_values(ascending = False)


In [9]:
df_940# - df_940_1

,holdings_today
stock_id,
2610,143539.000
2618,88204.000
9904,76425.000
2801,67852.000
2891,59127.000
2885,53699.000
2890,53446.000
2892,53274.944
2324,51269.000


## 使用stock_id查詢有哪些ETF持有量

In [10]:
df_delta[df_delta['stock_id'].str.contains('2353')]

,date_today,stock_id,weight(%)_today,holdings_today,etf,date_yesterday,weight(%)_yesterday,holdings_yesterday,holdings_diff
5,20260130.0,2353,1.51,48464.000,918,20260123.0,1.56,48464.000,0.000
6,20260122.0,2353,1.71,84872.000,929,20260120.0,1.73,86350.000,-1478.000
7,20260112.0,2353,0.38,4025.000,939,20260109.0,0.37,4037.000,-12.000
13,20251231.0,2353,0.73,144907.640,56,20251230.0,0.74,144718.872,188.768
12,20251229.0,2353,1.63,2302.376,932,20251226.0,1.62,2302.376,0.000


2542  1442  1808  2504  2520  5508  5522  9945

# 981A 982A

In [11]:
import pandas as pd
import os

# Define the ETFs to analyze
etfs_to_analyze = ['00982A', '00981A']

# Define the path format for the CSV files
path = "/content/drive/MyDrive/etf_daily_holdings/{}_{}.csv"

# Get today's date
date = pd.to_datetime('today', format='%Y%m%d').date()

# Initialize a list to store the results
results = []

for etf_id in etfs_to_analyze:
    # Find the most recent available data for today
    current_date = date
    today_file_path = path.format(current_date.strftime('%Y%m%d'), etf_id)
    today_df = pd.DataFrame()  # Initialize as empty

    while True:
        if os.path.isfile(today_file_path):
            today_df = pd.read_csv(today_file_path)
            print(f"Today's data for {etf_id} found at: {today_file_path}")
            break
        else:
            current_date -= pd.DateOffset(1)
            today_file_path = path.format(current_date.strftime('%Y%m%d'), etf_id)
            if (pd.Timestamp(date) - pd.Timestamp(current_date)).days > 30:
                print(f"No data found for {etf_id} in the last 30 days.")
                break

    if today_df.empty:
        continue

    # Find the most recent available data for yesterday
    yesterday_date = pd.Timestamp(current_date) - pd.DateOffset(1)
    yesterday_file_path = path.format(yesterday_date.strftime('%Y%m%d'), etf_id)
    yesterday_df = pd.DataFrame()  # Initialize as empty

    while True:
        if os.path.isfile(yesterday_file_path):
            yesterday_df = pd.read_csv(yesterday_file_path)
            print(f"Yesterday's data for {etf_id} found at: {yesterday_file_path}")
            break
        else:
            yesterday_date -= pd.DateOffset(1)
            yesterday_file_path = path.format(yesterday_date.strftime('%Y%m%d'), etf_id)
            if (pd.Timestamp(current_date) - pd.Timestamp(yesterday_date)).days > 30:
                print(f"No yesterday's data found for {etf_id} in the last 30 days.")
                break

    if yesterday_df.empty:
        continue

    # Merge the dataframes
    merged_df = pd.merge(today_df, yesterday_df, on=['stock_id', 'etf'], how='outer', suffixes=('_today', '_yesterday'))

    # Add date columns
    merged_df['date_today'] = current_date.strftime('%Y%m%d')
    merged_df['date_yesterday'] = yesterday_date.strftime('%Y%m%d')

    # Fill NaN holdings with 0 for difference calculation
    merged_df['holdings_today'] = merged_df['holdings_today'].fillna(0)
    merged_df['holdings_yesterday'] = merged_df['holdings_yesterday'].fillna(0)

    # Calculate holdings difference
    merged_df['holdings_diff'] = merged_df['holdings_today'] - merged_df['holdings_yesterday']

    # Reorder columns to match the example
    merged_df = merged_df[['date_today', 'stock_id', 'weight(%)_today', 'holdings_today', 'etf',
                           'date_yesterday', 'weight(%)_yesterday', 'holdings_yesterday', 'holdings_diff']]

    # Append to results
    results.append(merged_df)

# Concatenate all results, sort by holdings_diff descending then by etf ascending, and display the full dataframe
if results:
    results_df = pd.concat(results, ignore_index=True)
    results_df = results_df.sort_values(by=['holdings_diff', 'etf'], ascending=[False, True])
    results_df = results_df[results_df['stock_id'].str.len() == 4]
    results_df = results_df[results_df['holdings_diff'] != 0]
    display(results_df)
else:
    print("No data available for analysis.")

Today's data for 00982A found at: /content/drive/MyDrive/etf_daily_holdings/20260206_00982A.csv
Yesterday's data for 00982A found at: /content/drive/MyDrive/etf_daily_holdings/20260202_00982A.csv
Today's data for 00981A found at: /content/drive/MyDrive/etf_daily_holdings/20260206_00981A.csv
Yesterday's data for 00981A found at: /content/drive/MyDrive/etf_daily_holdings/20260202_00981A.csv


,date_today,stock_id,weight(%)_today,holdings_today,etf,date_yesterday,weight(%)_yesterday,holdings_yesterday,holdings_diff
70,20260206,1303,0.81,6261.0,00981A,20260202,0.00,1.0,6260.0
88,20260206,2449,0.63,1260.0,00981A,20260202,0.00,1.0,1259.0
93,20260206,3037,2.37,3974.0,00981A,20260202,2.01,2930.0,1044.0
15,20260206,2441,2.40,8347.0,00982A,20260202,2.45,7602.0,745.0
29,20260206,3231,0.56,1255.0,00982A,20260202,0.28,596.0,659.0
...,...,...,...,...,...,...,...,...,...
112,20260206,6415,NaN,0.0,00981A,20260202,0.00,1.0,-1.0
77,20260206,2308,5.21,2560.0,00981A,20260202,5.80,2625.0,-65.0
3,20260206,2059,0.56,55.0,00982A,20260202,1.77,155.0,-100.0
120,20260206,8299,5.36,1614.0,00981A,20260202,7.69,1908.0,-294.0
